# Experiment: Example Luptitude Light Curves

Objective:
- Reproduce the luptitude-space light-curve plotting style used in `optical_only/scripts/plot/plot_typical_luptitude_lightcurves.py`.
- Select representative KN light curves under the Baseline, Gold, and Silver observing strategies from the latest test dataset.
- Retain the representative negative examples from the existing optical-only contaminant dataset.
- Generate a compact multi-panel figure of example light curves that can be adjusted interactively in notebook cells.


In [1]:
import os
_BASE = os.environ.get('BASE_DIR', '/fred/oz016/bgao_kn')

from __future__ import annotations

import csv
import json
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence

import h5py
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd

BANDS: Sequence[str] = ("u", "g", "r", "i", "z", "Y")
BAND_COLORS: Dict[str, str] = {
    "u": "#3B82F6",
    "g": "#10B981",
    "r": "#EF4444",
    "i": "#F59E0B",
    "z": "#8B5CF6",
    "Y": "#6B7280",
}

BASE = Path(f"{_BASE}/gw-kn-multimodal")
POS_H5 = Path(f"{_BASE}/data/ALBEF_dataset/combined_dataset_astro_test.h5")
NEG_H5 = Path(f"{_BASE}/data/Optical_Negative_dataset/ELASTICC_negative_dataset.h5")
NEG_GROUP = "ELASTICC/optical_data"
STRATEGY_ARTIFACTS = {
    "bns": BASE / "kn_simulation/runs/bns_test/simulation_intermediates.h5",
    "nsbh": BASE / "kn_simulation/runs/nsbh_test/simulation_intermediates.h5",
}
STRATEGY_ORDER: Sequence[str] = ("Baseline", "Gold", "Silver")
POS_TIME_SCALE_DAYS = 100.0
OUTDIR = BASE / "figures" / "example_luptitude_lightcurves"
OUTDIR.mkdir(parents=True, exist_ok=True)
ASINH_MAG_FACTOR = 2.5 / np.log(10.0)
PLOT_TIME_WINDOW_DAYS = (-10.0, 20.0)

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.grid": True,
    "grid.color": "#d9d9d9",
    "grid.alpha": 0.55,
    "grid.linewidth": 0.8,
    "axes.spines.top": True,
    "axes.spines.right": True,
    "axes.edgecolor": "black",
    "axes.linewidth": 0.9,
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "axes.titlesize": 20,
    "axes.labelsize": 19,
    "xtick.labelsize": 17,
    "ytick.labelsize": 17,
    "legend.frameon": False,
})

POS_H5, NEG_H5, OUTDIR

(PosixPath('/fred/oz016/bgao_kn/data/ALBEF_dataset/combined_dataset_astro_test.h5'),
 PosixPath('/fred/oz016/bgao_kn/data/Optical_Negative_dataset/ELASTICC_negative_dataset.h5'),
 PosixPath('/fred/oz016/bgao_kn/gw-kn-multimodal/figures/example_luptitude_lightcurves'))

## Plan

- Read the latest preprocessed KN test HDF5 dataset and the existing optical-only contaminant dataset.
- Join KN events to their HDF5 observation plans and select one typical light curve for each of Baseline, Gold, and Silver.
- Use the same robust-typicality ranking idea as the reference script; retain the existing negative-sample selection.
- Plot per-band luptitude measurements with error bars and an inverted y-axis.
- Save one combined figure and a CSV manifest of the selected examples.


In [2]:
@dataclass
class DatasetConfig:
    class_name: str
    h5_path: Path
    group: str
    unique_mode: Optional[str]


@dataclass
class SelectionResult:
    class_name: str
    sample_index: int
    meta_score: float
    typical_score: float
    source_label: str
    source_id: str
    parent_event_idx: Optional[int]
    n_obs: int
    n_det_snr5: int
    n_bands: int
    t_span_scaled: float
    t_span_days: float
    mean_lupt: float
    std_lupt: float
    amp_lupt: float
    mean_err: float
    plot_path: Optional[Path] = None


def decode_scalar(value) -> str:
    if isinstance(value, (bytes, np.bytes_)):
        return value.decode("utf-8", errors="ignore")
    return str(value)


def robust_scales(features: np.ndarray) -> np.ndarray:
    med = np.nanmedian(features, axis=0)
    mad = np.nanmedian(np.abs(features - med), axis=0)
    std = np.nanstd(features, axis=0)
    scale = np.where(mad > 1.0e-6, mad, np.where(std > 1.0e-6, std, 1.0))
    return scale.astype(np.float64, copy=False)


def robust_scores(features: np.ndarray) -> np.ndarray:
    center = np.nanmedian(features, axis=0)
    scale = robust_scales(features)
    z = (features - center) / scale
    return np.sqrt(np.sum(np.square(z), axis=1, dtype=np.float64), dtype=np.float64)


def compute_lc_summary_for_indices(
    grp: h5py.Group,
    sample_indices: np.ndarray,
    time_window_scaled: Optional[tuple[float, float]] = None,
) -> np.ndarray:
    sorted_pos = np.argsort(sample_indices)
    sorted_indices = np.asarray(sample_indices[sorted_pos], dtype=np.int64)

    values = np.asarray(grp["values"][sorted_indices], dtype=np.float32)
    errors = np.asarray(grp["errors"][sorted_indices], dtype=np.float32)
    masks = np.asarray(grp["masks"][sorted_indices], dtype=np.float32) > 0.0

    if time_window_scaled is not None:
        times_scaled = np.asarray(grp["times"][sorted_indices], dtype=np.float64)
        start_scaled, end_scaled = map(float, time_window_scaled)
        in_window = np.isfinite(times_scaled) & (times_scaled >= start_scaled) & (times_scaled <= end_scaled)
        masks = masks & in_window[:, :, None]

    counts = masks.sum(axis=(1, 2)).astype(np.float32)
    counts_safe = np.where(counts > 0.0, counts, 1.0)

    values_masked = np.where(masks, values, 0.0)
    errors_masked = np.where(masks, errors, 0.0)

    mean_lupt = values_masked.sum(axis=(1, 2), dtype=np.float64) / counts_safe
    sq_mean = np.square(values_masked, dtype=np.float64).sum(axis=(1, 2), dtype=np.float64) / counts_safe
    std_lupt = np.sqrt(np.maximum(0.0, sq_mean - np.square(mean_lupt, dtype=np.float64)))
    max_vals = np.max(np.where(masks, values, -np.inf), axis=(1, 2)).astype(np.float64)
    min_vals = np.min(np.where(masks, values, np.inf), axis=(1, 2)).astype(np.float64)
    amp_lupt = np.where(counts > 0.0, max_vals - min_vals, 0.0)
    mean_err = errors_masked.sum(axis=(1, 2), dtype=np.float64) / counts_safe

    mean_lupt = np.where(counts > 0.0, mean_lupt, 0.0)
    std_lupt = np.where(counts > 0.0, std_lupt, 0.0)
    mean_err = np.where(counts > 0.0, mean_err, 0.0)

    summary_sorted = np.column_stack([mean_lupt, std_lupt, amp_lupt, mean_err]).astype(np.float64, copy=False)
    summary = np.empty_like(summary_sorted)
    summary[sorted_pos] = summary_sorted
    return summary


def compute_window_observation_meta_for_indices(
    grp: h5py.Group,
    sample_indices: np.ndarray,
    time_window_scaled: Optional[tuple[float, float]],
) -> tuple[np.ndarray, np.ndarray]:
    sorted_pos = np.argsort(sample_indices)
    sorted_indices = np.asarray(sample_indices[sorted_pos], dtype=np.int64)

    masks = np.asarray(grp["masks"][sorted_indices], dtype=np.float32) > 0.0
    if time_window_scaled is not None:
        times_scaled = np.asarray(grp["times"][sorted_indices], dtype=np.float64)
        start_scaled, end_scaled = map(float, time_window_scaled)
        in_window = np.isfinite(times_scaled) & (times_scaled >= start_scaled) & (times_scaled <= end_scaled)
        masks = masks & in_window[:, :, None]

    n_obs_sorted = masks.sum(axis=(1, 2)).astype(np.float64)
    n_bands_sorted = np.count_nonzero(masks.sum(axis=1) > 0, axis=1).astype(np.float64)

    n_obs = np.empty_like(n_obs_sorted)
    n_bands = np.empty_like(n_bands_sorted)
    n_obs[sorted_pos] = n_obs_sorted
    n_bands[sorted_pos] = n_bands_sorted
    return n_obs, n_bands


def dataset_time_scale(h5f: h5py.File) -> float:
    if "time_scale_divisor_days" in h5f.attrs:
        return float(h5f.attrs["time_scale_divisor_days"])
    if Path(h5f.filename).resolve() == POS_H5.resolve():
        return POS_TIME_SCALE_DAYS
    return 1.0


def load_selection_metadata(grp: h5py.Group) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    required = ("meta_n_det", "meta_n_bands", "meta_t_span")
    if all(name in grp for name in required):
        return tuple(np.asarray(grp[name][:], dtype=np.float64) for name in required)

    masks = np.asarray(grp["masks"][:], dtype=np.float32) > 0.0
    times = np.asarray(grp["times"][:], dtype=np.float64)
    observed_slots = masks.any(axis=2) & np.isfinite(times)
    n_obs = masks.sum(axis=(1, 2)).astype(np.float64)
    n_bands = np.count_nonzero(masks.sum(axis=1) > 0, axis=1).astype(np.float64)
    t_min = np.min(np.where(observed_slots, times, np.inf), axis=1)
    t_max = np.max(np.where(observed_slots, times, -np.inf), axis=1)
    t_span = np.where(observed_slots.any(axis=1), t_max - t_min, 0.0)
    return n_obs, n_bands, t_span


def load_positive_strategy_labels(h5f: h5py.File, grp: h5py.Group) -> np.ndarray:
    gw = h5f["events/gw_data"]
    source_types = np.asarray([decode_scalar(value) for value in gw["source_type"][:]], dtype=object)
    simulation_ids = np.asarray(gw["simulation_id"][:], dtype=np.int64)
    has_kn = np.asarray(gw["has_kn"][:], dtype=bool)

    plan_labels: Dict[tuple[str, int], str] = {}
    for source_type, artifact_path in STRATEGY_ARTIFACTS.items():
        with h5py.File(artifact_path, "r") as artifact:
            artifact_ids = np.asarray(artifact["events/simulation_id"][:], dtype=np.int64)
            plans = artifact["events/observation_plan_json"][:]
            for simulation_id, raw_plan in zip(artifact_ids, plans):
                plan = json.loads(decode_scalar(raw_plan))
                has_too = (
                    plan.get("mode") == "baseline_plus_too"
                    and int(plan.get("too_visits_written", 0) or 0) > 0
                )
                strategy = plan.get("strategy")
                if not has_too:
                    label = "Baseline"
                elif strategy in {"gold", "gold_five_filter"}:
                    label = "Gold"
                elif strategy == "silver":
                    label = "Silver"
                else:
                    raise ValueError(f"Unrecognized ToO strategy: {strategy!r}")
                plan_labels[(source_type, int(simulation_id))] = label

    event_labels = np.asarray(
        [plan_labels.get((source_type, int(simulation_id)), "") for source_type, simulation_id in zip(source_types, simulation_ids)],
        dtype=object,
    )
    if np.any(event_labels == ""):
        raise ValueError("At least one GW event has no observation-strategy label")

    parent_indices = np.asarray(grp["parent_gw_idx"][:], dtype=np.int64)
    sample_labels = event_labels[parent_indices]
    sample_labels[~has_kn[parent_indices]] = ""
    return sample_labels


In [3]:
def compute_observation_and_detection_counts(
    h5f: h5py.File,
    grp: h5py.Group,
    sample_index: int,
    time_window_scaled: Optional[tuple[float, float]] = None,
) -> tuple[int, int, int]:
    values = np.asarray(grp["values"][sample_index], dtype=np.float64)
    errors = np.asarray(grp["errors"][sample_index], dtype=np.float64)
    masks = np.asarray(grp["masks"][sample_index], dtype=np.float32) > 0.0

    if time_window_scaled is not None:
        times_scaled = np.asarray(grp["times"][sample_index], dtype=np.float64)
        start_scaled, end_scaled = map(float, time_window_scaled)
        in_window = np.isfinite(times_scaled) & (times_scaled >= start_scaled) & (times_scaled <= end_scaled)
        masks = masks & in_window[:, None]

    n_obs = int(np.count_nonzero(masks))
    n_bands = int(np.count_nonzero(masks.sum(axis=0) > 0))
    if n_obs < 1:
        return 0, 0, 0

    lupt_b_njy = np.asarray(h5f.attrs["lupt_b_njy"], dtype=np.float64).reshape(-1)
    psfflux_zp = float(h5f.attrs["psfflux_zp"])
    snr_threshold = float(h5f.attrs.get("snr_threshold", 5.0))

    band_grid = np.broadcast_to(np.arange(values.shape[1], dtype=np.int64), values.shape)
    valid = masks & np.isfinite(values) & np.isfinite(errors) & (errors > 0.0)
    if not np.any(valid):
        return n_obs, 0, n_bands

    m_lupt = values[valid]
    sigma_lupt = errors[valid]
    band_idx = band_grid[valid]
    b_valid = lupt_b_njy[band_idx]

    asinh_arg = (psfflux_zp - m_lupt) / ASINH_MAG_FACTOR - np.log(b_valid)
    f_psf = 2.0 * b_valid * np.sinh(asinh_arg)
    sigma_psf = sigma_lupt * np.sqrt((f_psf * f_psf) + (2.0 * b_valid) ** 2) / ASINH_MAG_FACTOR

    snr = np.full(f_psf.shape, -np.inf, dtype=np.float64)
    finite = np.isfinite(f_psf) & np.isfinite(sigma_psf) & (sigma_psf > 0.0)
    snr[finite] = f_psf[finite] / sigma_psf[finite]
    n_det_snr5 = int(np.count_nonzero(snr > snr_threshold))
    return n_obs, n_det_snr5, n_bands


def select_typical_subset(
    *,
    h5f: h5py.File,
    grp: h5py.Group,
    cfg: DatasetConfig,
    time_scale: float,
    n_obs_all: np.ndarray,
    n_bands_all: np.ndarray,
    t_span_scaled_all: np.ndarray,
    candidate_indices: np.ndarray,
    max_results: int,
    shortlist_size: int,
    display_time_window_scaled: Optional[tuple[float, float]] = None,
    selection_time_window_scaled: Optional[tuple[float, float]] = None,
    source_label_override: Optional[str] = None,
) -> List[SelectionResult]:
    candidate_indices = np.asarray(candidate_indices, dtype=np.int64)
    if candidate_indices.size < 1:
        return []

    meta_features = np.column_stack([
        n_obs_all[candidate_indices],
        n_bands_all[candidate_indices],
        t_span_scaled_all[candidate_indices],
    ]).astype(np.float64, copy=False)
    meta_scores_subset = robust_scores(meta_features)
    shortlist_n = min(candidate_indices.size, max(int(shortlist_size), int(max_results) * 32))
    shortlist_local = np.argsort(meta_scores_subset)[:shortlist_n]
    shortlist = candidate_indices[shortlist_local]

    lc_summary = compute_lc_summary_for_indices(grp, shortlist, time_window_scaled=selection_time_window_scaled)
    combined_features = np.column_stack([meta_features[shortlist_local], lc_summary]).astype(np.float64, copy=False)
    combined_scores = robust_scores(combined_features)
    order = np.argsort(combined_scores)

    unique_values = None
    if cfg.unique_mode == "parent_event_idx":
        if "parent_event_idx" in grp:
            unique_values = np.asarray(grp["parent_event_idx"][:], dtype=np.int64)[shortlist]
        elif "parent_gw_idx" in grp:
            unique_values = np.asarray(grp["parent_gw_idx"][:], dtype=np.int64)[shortlist]

    selected_short_positions = []
    seen_unique = set()
    for short_pos in order.tolist():
        if unique_values is not None:
            unique_key = int(unique_values[short_pos])
            if unique_key in seen_unique:
                continue
            seen_unique.add(unique_key)
        selected_short_positions.append(short_pos)
        if len(selected_short_positions) >= max_results:
            break

    results = []
    for short_pos in selected_short_positions:
        sample_idx = int(shortlist[short_pos])
        parent_event_idx = None
        source_label = ""
        source_id = ""

        if cfg.class_name == "positive":
            source_label = source_label_override or ""
            if "parent_event_idx" in grp:
                parent_event_idx = int(grp["parent_event_idx"][sample_idx])
            elif "parent_gw_idx" in grp:
                parent_event_idx = int(grp["parent_gw_idx"][sample_idx])
        else:
            source_label = source_label_override or (decode_scalar(grp["types"][sample_idx]) if "types" in grp else "unknown")

        n_obs, n_det_snr5, n_bands = compute_observation_and_detection_counts(
            h5f=h5f,
            grp=grp,
            sample_index=sample_idx,
            time_window_scaled=display_time_window_scaled,
        )
        results.append(
            SelectionResult(
                class_name=cfg.class_name,
                sample_index=sample_idx,
                meta_score=float(meta_scores_subset[shortlist_local[short_pos]]),
                typical_score=float(combined_scores[short_pos]),
                source_label=source_label,
                source_id=source_id,
                parent_event_idx=parent_event_idx,
                n_obs=int(n_obs),
                n_det_snr5=int(n_det_snr5),
                n_bands=int(n_bands),
                t_span_scaled=float(t_span_scaled_all[sample_idx]),
                t_span_days=float(t_span_scaled_all[sample_idx] * time_scale),
                mean_lupt=float(lc_summary[short_pos, 0]),
                std_lupt=float(lc_summary[short_pos, 1]),
                amp_lupt=float(lc_summary[short_pos, 2]),
                mean_err=float(lc_summary[short_pos, 3]),
            )
        )

    results.sort(key=lambda item: item.typical_score)
    return results


def select_distinctive_subset(
    *,
    h5f: h5py.File,
    grp: h5py.Group,
    cfg: DatasetConfig,
    time_scale: float,
    n_obs_all: np.ndarray,
    n_bands_all: np.ndarray,
    t_span_scaled_all: np.ndarray,
    candidate_indices: np.ndarray,
    max_results: int,
    shortlist_size: int,
    display_time_window_scaled: Optional[tuple[float, float]] = None,
    selection_time_window_scaled: Optional[tuple[float, float]] = None,
    source_label_override: Optional[str] = None,
) -> List[SelectionResult]:
    """Select samples with the most prominent light-curve features.

    Ranks candidates by a prominence score that weights luptitude amplitude (×2),
    number of observed bands, and total observation count. Intended for transient
    types (e.g. SN, TDE) where we want the clearest, most visually striking examples.
    """
    candidate_indices = np.asarray(candidate_indices, dtype=np.int64)
    if candidate_indices.size < 1:
        return []

    pre_score = n_obs_all[candidate_indices].astype(np.float64) * n_bands_all[candidate_indices].astype(np.float64)
    shortlist_n = min(candidate_indices.size, max(int(shortlist_size), int(max_results) * 32))
    shortlist_local = np.argsort(-pre_score)[:shortlist_n]
    shortlist = candidate_indices[shortlist_local]

    lc_summary = compute_lc_summary_for_indices(grp, shortlist, time_window_scaled=selection_time_window_scaled)
    amp = lc_summary[:, 2].astype(np.float64)
    if selection_time_window_scaled is not None:
        n_o, n_b = compute_window_observation_meta_for_indices(
            grp,
            shortlist,
            time_window_scaled=selection_time_window_scaled,
        )
    else:
        n_b = n_bands_all[shortlist].astype(np.float64)
        n_o = n_obs_all[shortlist].astype(np.float64)

    def safe_norm(arr: np.ndarray) -> np.ndarray:
        lo, hi = arr.min(), arr.max()
        return (arr - lo) / (hi - lo) if hi - lo > 1e-9 else np.ones_like(arr)

    prominence = 2.0 * safe_norm(amp) + safe_norm(n_b) + safe_norm(n_o)
    order = np.argsort(-prominence)

    meta_features = np.column_stack([
        n_obs_all[candidate_indices],
        n_bands_all[candidate_indices],
        t_span_scaled_all[candidate_indices],
    ]).astype(np.float64, copy=False)
    meta_scores_subset = robust_scores(meta_features)

    selected_short_positions: List[int] = []
    for short_pos in order.tolist():
        selected_short_positions.append(int(short_pos))
        if len(selected_short_positions) >= max_results:
            break

    results = []
    for short_pos in selected_short_positions:
        sample_idx = int(shortlist[short_pos])
        source_label = source_label_override or (decode_scalar(grp["types"][sample_idx]) if "types" in grp else "unknown")
        n_obs, n_det_snr5, n_bands = compute_observation_and_detection_counts(
            h5f=h5f,
            grp=grp,
            sample_index=sample_idx,
            time_window_scaled=display_time_window_scaled,
        )
        results.append(
            SelectionResult(
                class_name=cfg.class_name,
                sample_index=sample_idx,
                meta_score=float(meta_scores_subset[shortlist_local[short_pos]]),
                typical_score=float(-prominence[short_pos]),
                source_label=source_label,
                source_id="",
                parent_event_idx=None,
                n_obs=int(n_obs),
                n_det_snr5=int(n_det_snr5),
                n_bands=int(n_bands),
                t_span_scaled=float(t_span_scaled_all[sample_idx]),
                t_span_days=float(t_span_scaled_all[sample_idx] * time_scale),
                mean_lupt=float(lc_summary[short_pos, 0]),
                std_lupt=float(lc_summary[short_pos, 1]),
                amp_lupt=float(lc_summary[short_pos, 2]),
                mean_err=float(lc_summary[short_pos, 3]),
            )
        )

    return results


In [4]:
# Transient types for which we pick the most visually distinctive example
# (high amplitude, many bands, many detections) rather than the most typical one.
DISTINCTIVE_TYPES: set[str] = {"SN", "TDE"}
WINDOW_DISTINCTIVE_TYPES: set[str] = {"TDE"}


def select_typical_samples(cfg: DatasetConfig, samples_per_class: int = 3, shortlist_size: int = 4096,
                           negative_selection_mode: str = "per_type_typical", random_pool_size: int = 1024,
                           random_seed: int = 42, min_positive_detections: int = 2) -> List[SelectionResult]:
    with h5py.File(cfg.h5_path, "r") as h5f:
        grp = h5f[cfg.group]
        time_scale = dataset_time_scale(h5f)
        display_time_window_scaled = tuple(float(x / time_scale) for x in PLOT_TIME_WINDOW_DAYS)
        n_obs_meta, n_bands, t_span_scaled = load_selection_metadata(grp)
        n_total = int(grp["values"].shape[0])

        if cfg.class_name == "positive":
            strategy_labels = load_positive_strategy_labels(h5f, grp)
            selected = []
            for strategy in STRATEGY_ORDER:
                candidate_indices = np.flatnonzero(strategy_labels == strategy)
                candidate_results = select_typical_subset(
                    h5f=h5f,
                    grp=grp,
                    cfg=cfg,
                    time_scale=time_scale,
                    n_obs_all=n_obs_meta,
                    n_bands_all=n_bands,
                    t_span_scaled_all=t_span_scaled,
                    candidate_indices=candidate_indices,
                    max_results=min(36, candidate_indices.size),
                    shortlist_size=min(shortlist_size, candidate_indices.size),
                    display_time_window_scaled=display_time_window_scaled,
                    source_label_override=strategy,
                )
                filtered = [result for result in candidate_results if result.n_det_snr5 >= min_positive_detections]
                if not filtered:
                    filtered = [result for result in candidate_results if result.n_det_snr5 > 0]
                if not filtered:
                    filtered = candidate_results
                if not filtered:
                    raise ValueError(f"No positive KN light curve found for {strategy}")
                selected.append(filtered[0])
            return selected

        if cfg.class_name == "negative" and "types" in grp and negative_selection_mode == "per_type_typical":
            type_labels = np.asarray([decode_scalar(v) for v in grp["types"][:]], dtype=object)
            unique_labels, counts = np.unique(type_labels, return_counts=True)
            label_order = [str(lbl) for lbl in unique_labels[np.argsort(-counts, kind="stable")]]
            results = []
            for label in label_order[:samples_per_class]:
                type_indices = np.flatnonzero(type_labels == label)
                if label in DISTINCTIVE_TYPES:
                    selection_time_window_scaled = display_time_window_scaled if label in WINDOW_DISTINCTIVE_TYPES else None
                    results.extend(
                        select_distinctive_subset(
                            h5f=h5f,
                            grp=grp,
                            cfg=cfg,
                            time_scale=time_scale,
                            n_obs_all=n_obs_meta,
                            n_bands_all=n_bands,
                            t_span_scaled_all=t_span_scaled,
                            candidate_indices=type_indices,
                            max_results=1,
                            shortlist_size=shortlist_size,
                            display_time_window_scaled=display_time_window_scaled,
                            selection_time_window_scaled=selection_time_window_scaled,
                            source_label_override=label,
                        )
                    )
                else:
                    results.extend(
                        select_typical_subset(
                            h5f=h5f,
                            grp=grp,
                            cfg=cfg,
                            time_scale=time_scale,
                            n_obs_all=n_obs_meta,
                            n_bands_all=n_bands,
                            t_span_scaled_all=t_span_scaled,
                            candidate_indices=type_indices,
                            max_results=1,
                            shortlist_size=shortlist_size,
                            display_time_window_scaled=display_time_window_scaled,
                            source_label_override=label,
                        )
                    )
            return results

        return select_typical_subset(
            h5f=h5f,
            grp=grp,
            cfg=cfg,
            time_scale=time_scale,
            n_obs_all=n_obs_meta,
            n_bands_all=n_bands,
            t_span_scaled_all=t_span_scaled,
            candidate_indices=np.arange(n_total, dtype=np.int64),
            max_results=samples_per_class,
            shortlist_size=shortlist_size,
            display_time_window_scaled=display_time_window_scaled,
        )


def extract_sample_plot_data(h5_path: Path, group: str, sample_index: int, plot_time_unit: str = "days") -> Dict[str, object]:
    with h5py.File(h5_path, "r") as h5f:
        grp = h5f[group]
        values = np.asarray(grp["values"][sample_index], dtype=np.float64)
        errors = np.asarray(grp["errors"][sample_index], dtype=np.float64)
        masks = np.asarray(grp["masks"][sample_index], dtype=np.float32) > 0.0
        times_scaled = np.asarray(grp["times"][sample_index], dtype=np.float64)
        time_scale = dataset_time_scale(h5f)
        if plot_time_unit == "days":
            times = times_scaled * time_scale
            x_label = "Time since first detection (days)"
        else:
            times = times_scaled
            x_label = "Scaled time since first detection"

        lupt_b_njy = np.asarray(h5f.attrs["lupt_b_njy"], dtype=np.float64).reshape(-1)
        psfflux_zp = float(h5f.attrs["psfflux_zp"])
        snr_threshold = float(h5f.attrs.get("snr_threshold", 5.0))

        band_grid = np.broadcast_to(np.arange(values.shape[1], dtype=np.int64), values.shape)
        snr = np.full(values.shape, np.nan, dtype=np.float64)
        valid = masks & np.isfinite(values) & np.isfinite(errors) & (errors > 0.0)
        if np.any(valid):
            m_lupt = values[valid]
            sigma_lupt = errors[valid]
            band_idx = band_grid[valid]
            b_valid = lupt_b_njy[band_idx]
            asinh_arg = (psfflux_zp - m_lupt) / ASINH_MAG_FACTOR - np.log(b_valid)
            f_psf = 2.0 * b_valid * np.sinh(asinh_arg)
            sigma_psf = sigma_lupt * np.sqrt((f_psf * f_psf) + (2.0 * b_valid) ** 2) / ASINH_MAG_FACTOR
            finite = np.isfinite(f_psf) & np.isfinite(sigma_psf) & (sigma_psf > 0.0)
            snr_valid = np.full(m_lupt.shape, np.nan, dtype=np.float64)
            snr_valid[finite] = f_psf[finite] / sigma_psf[finite]
            snr[valid] = snr_valid

        return {
            "values": values,
            "errors": errors,
            "masks": masks,
            "times": times,
            "x_label": x_label,
            "snr": snr,
            "snr_threshold": snr_threshold,
        }




In [5]:


def add_panel_labels_below_grid(axes, fontsize=18):
    visible_axes = [ax for ax in axes.flat if ax.axison]
    for idx, ax in enumerate(visible_axes):
        ax.text(
            0.5,
            -0.24,
            f"({chr(ord('a') + idx)})",
            transform=ax.transAxes,
            ha="center",
            va="top",
            fontsize=fontsize,
            clip_on=False,
        )

def style_closed_axes(ax):
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.9)
        spine.set_color("black")
    ax.tick_params(axis="both", which="both", direction="out", top=False, right=False, length=4, width=0.8)
    ax.grid(True, color="#d9d9d9", alpha=0.55, linewidth=0.8)


def set_luptitude_ylim(ax, values, errors):
    values = np.asarray(values, dtype=float)
    errors = np.asarray(errors, dtype=float)
    finite = np.isfinite(values)
    if not np.any(finite):
        ax.set_ylim(30.5, 22.0)
        return
    lo = float(np.nanmin(values[finite]))
    hi = float(np.nanmax(values[finite]))
    err_hi = values[finite] + np.nan_to_num(errors[finite], nan=0.0)
    hi = max(hi, float(np.nanpercentile(err_hi, 90)))
    pad = max(0.35, 0.08 * max(hi - lo, 1.0))
    ax.set_ylim(hi + pad, lo - pad)


def build_band_legend_handles() -> List[Line2D]:
    handles = []
    for band_name in BANDS:
        handles.append(
            Line2D(
                [0], [0], marker="o", linestyle="none", markersize=7.0,
                markerfacecolor=BAND_COLORS[band_name], markeredgecolor=BAND_COLORS[band_name],
                label=f"{band_name}-band",
            )
        )
    return handles


def plot_single_sample(ax: plt.Axes, result: SelectionResult, sample_data: Dict[str, object], rank: int) -> None:
    values = sample_data["values"]
    errors = sample_data["errors"]
    masks = sample_data["masks"]
    times = sample_data["times"]
    snr = sample_data["snr"]
    snr_threshold = float(sample_data["snr_threshold"])

    plotted_values = []
    plotted_errors = []

    for band_idx, band_name in enumerate(BANDS):
        valid = masks[:, band_idx]
        if not np.any(valid):
            continue
        det_mask = valid & np.isfinite(snr[:, band_idx]) & (snr[:, band_idx] >= snr_threshold)
        nondet_mask = valid & (~det_mask)

        for mask, filled in ((det_mask, True), (nondet_mask, False)):
            if not np.any(mask):
                continue
            x = np.asarray(times[mask], dtype=np.float64)
            y = np.asarray(values[mask, band_idx], dtype=np.float64)
            yerr = np.asarray(errors[mask, band_idx], dtype=np.float64)
            sort_idx = np.argsort(x)
            plotted_values.append(y)
            plotted_errors.append(yerr)
            markerfacecolor = BAND_COLORS[band_name] if filled else "none"
            ax.errorbar(
                x[sort_idx],
                y[sort_idx],
                yerr=yerr[sort_idx],
                fmt="o",
                ms=4.2,
                mew=1.2,
                lw=0.0,
                elinewidth=1.0,
                capsize=2.2,
                alpha=0.95,
                color=BAND_COLORS[band_name],
                markerfacecolor=markerfacecolor,
                markeredgecolor=BAND_COLORS[band_name],
            )

    ax.axvline(0.0, color="#4b5563", lw=1.0, ls="--", alpha=0.9)
    ax.set_xlim(*PLOT_TIME_WINDOW_DAYS)
    ax.set_xlabel("Time since first detection (days)")
    ax.set_ylabel("Luptitude")
    transient_label = f"KN ({result.source_label})" if result.class_name == "positive" else result.source_label
    baseline_panel = result.class_name == "positive" and result.source_label == "Baseline"
    label_x = 0.97 if baseline_panel else 0.03
    label_ha = "right" if baseline_panel else "left"
    ax.text(label_x, 0.95, transient_label, transform=ax.transAxes, ha=label_ha, va="top", fontsize=18)
    ax.invert_yaxis()
    if plotted_values:
        set_luptitude_ylim(ax, np.concatenate(plotted_values), np.concatenate(plotted_errors))
    else:
        ax.set_ylim(30.5, 22.0)
    style_closed_axes(ax)


def save_combined_plot(results_by_class: Dict[str, List[SelectionResult]], configs: Dict[str, DatasetConfig],
                       output_dir: Path, plot_time_unit: str = "days", dpi: int = 300) -> Path:
    samples_per_class = max(len(v) for v in results_by_class.values())
    ncols = samples_per_class
    nrows = 2
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.4 * ncols, 5.6 * nrows), squeeze=False, sharey=False)
    band_legend_handles = build_band_legend_handles()

    for row, class_name in enumerate(("positive", "negative")):
        results = results_by_class.get(class_name, [])
        for col in range(samples_per_class):
            ax = axes[row, col]
            if col >= len(results):
                ax.axis("off")
                continue
            result = results[col]
            sample_data = extract_sample_plot_data(
                h5_path=configs[class_name].h5_path,
                group=configs[class_name].group,
                sample_index=result.sample_index,
                plot_time_unit=plot_time_unit,
            )
            plot_single_sample(ax, result, sample_data, rank=col + 1)

    fig.legend(
        band_legend_handles,
        [h.get_label() for h in band_legend_handles],
        loc="upper center",
        bbox_to_anchor=(0.5, 0.95),
        ncol=len(BANDS),
        frameon=False,
        fontsize=19,
    )
    add_panel_labels_below_grid(axes, fontsize=19)
    fig.tight_layout(rect=(0, 0.07, 1, 0.89), h_pad=0.55, w_pad=1.8)

    out_path = output_dir / "example_luptitude_lightcurves.png"
    fig.savefig(out_path, dpi=dpi, bbox_inches="tight")
    fig.savefig(output_dir / "example_luptitude_lightcurves.pdf", bbox_inches="tight")
    plt.close(fig)
    return out_path


def write_manifest(manifest_path: Path, results_by_class: Dict[str, List[SelectionResult]]) -> None:
    fieldnames = [
        "class_name", "rank", "sample_index", "source_label", "parent_event_idx", "n_obs", "n_det_snr5",
        "n_bands", "t_span_days", "mean_lupt", "std_lupt", "amp_lupt", "mean_err", "typical_score",
    ]
    with manifest_path.open("w", newline="", encoding="utf-8") as fp:
        writer = csv.DictWriter(fp, fieldnames=fieldnames)
        writer.writeheader()
        for class_name in ("positive", "negative"):
            for rank, result in enumerate(results_by_class.get(class_name, []), start=1):
                writer.writerow({
                    "class_name": result.class_name,
                    "rank": rank,
                    "sample_index": result.sample_index,
                    "source_label": result.source_label,
                    "parent_event_idx": "" if result.parent_event_idx is None else result.parent_event_idx,
                    "n_obs": result.n_obs,
                    "n_det_snr5": result.n_det_snr5,
                    "n_bands": result.n_bands,
                    "t_span_days": f"{result.t_span_days:.3f}",
                    "mean_lupt": f"{result.mean_lupt:.4f}",
                    "std_lupt": f"{result.std_lupt:.4f}",
                    "amp_lupt": f"{result.amp_lupt:.4f}",
                    "mean_err": f"{result.mean_err:.4f}",
                    "typical_score": f"{result.typical_score:.4f}",
                })

## Choose Representative Samples

The three positive KN examples are selected separately from the Baseline, Gold, and Silver observation-strategy subsets in the latest test HDF5 data, with `Ndet(SNR>5) >= 2` enforced first. Strategy labels are joined through each GW event's `source_type` and `simulation_id` to the corresponding HDF5 observation plan. The existing negative-sample selection and all plotting settings are unchanged.


In [6]:
configs = {
    "positive": DatasetConfig(
        class_name="positive",
        h5_path=POS_H5,
        group="events/optical_data",
        unique_mode="parent_event_idx",
    ),
    "negative": DatasetConfig(
        class_name="negative",
        h5_path=NEG_H5,
        group=NEG_GROUP,
        unique_mode=None,
    ),
}

results_by_class = {
    class_name: select_typical_samples(cfg=cfg, samples_per_class=3, shortlist_size=4096)
    for class_name, cfg in configs.items()
}

pd.DataFrame([
    {
        "class": result.class_name,
        "sample_index": result.sample_index,
        "label": result.source_label,
        "n_obs": result.n_obs,
        "n_det_snr5": result.n_det_snr5,
        "span_days": round(result.t_span_days, 2),
        "score": round(result.typical_score, 3),
    }
    for class_name in ("positive", "negative")
    for result in results_by_class[class_name]
])

,class,sample_index,label,n_obs,n_det_snr5,span_days,score
0,positive,58295,Baseline,12,2,82.84,0.551
1,positive,25911,Gold,9,9,57.97,0.652
2,positive,62059,Silver,12,5,59.99,0.714
3,negative,877145,SN,14,14,83.86,-3.351
4,negative,1404914,TDE,17,13,88.86,-3.552
5,negative,20688,AGN,16,3,75.80,0.197


## Plot Example Light Curves


In [7]:
combined_plot = save_combined_plot(
    results_by_class=results_by_class,
    configs=configs,
    output_dir=OUTDIR,
    plot_time_unit="days",
    dpi=300,
)
manifest_path = OUTDIR / "example_luptitude_lightcurves_manifest.csv"
write_manifest(manifest_path, results_by_class)

{
    "combined_plot": str(combined_plot),
    "manifest": str(manifest_path),
}

{'combined_plot': '/fred/oz016/bgao_kn/gw-kn-multimodal/figures/example_luptitude_lightcurves/example_luptitude_lightcurves.png',
 'manifest': '/fred/oz016/bgao_kn/gw-kn-multimodal/figures/example_luptitude_lightcurves/example_luptitude_lightcurves_manifest.csv'}

In [8]:
result = {
    "positive_examples": len(results_by_class["positive"]),
    "negative_examples": len(results_by_class["negative"]),
    "combined_plot": str(combined_plot),
    "manifest": str(manifest_path),
}
result

{'positive_examples': 3,
 'negative_examples': 3,
 'combined_plot': '/fred/oz016/bgao_kn/gw-kn-multimodal/figures/example_luptitude_lightcurves/example_luptitude_lightcurves.png',
 'manifest': '/fred/oz016/bgao_kn/gw-kn-multimodal/figures/example_luptitude_lightcurves/example_luptitude_lightcurves_manifest.csv'}

## Results

- This notebook follows the light-curve plotting style of the reference script without changing the base figure settings.
- The first row shows typical KN light curves for Baseline, Gold, and Silver; the second row retains the representative SN, TDE, and AGN samples.
- The output is a combined multi-panel luptitude light-curve figure plus a manifest of the selected samples and strategy labels.
- You can change `samples_per_class`, `plot_time_unit`, or the output directory directly in the cells above.
